<a href="https://colab.research.google.com/github/ymuto0302/PJ2025/blob/main/ConvAE_CIFAR10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIFAR-10 を題材とした畳み込み自己符号化器（Convolutional Autoencoder）のPyTorch 実装

全結合層の代わりに畳み込み層（Conv2d）と転置畳み込み層（ConvTranspose2d）を用いる。
畳み込みは画像の「空間的な構造」（隣り合う画素の関係，エッジ，模様など）をそのまま扱えるため，画像の特徴抽出に適している。

```
ネットワーク構造（入力 3 x 32 x 32）:

エンコーダ（空間を縮め，チャンネルを増やす）:
    3x32x32 -> 32x16x16 -> 64x8x8 -> 128x4x4 -> [平坦化] -> 潜在表現(latent_dim)
デコーダ（エンコーダの鏡像）:
   潜在表現 -> 128x4x4 -> 64x8x8 -> 32x16x16 -> 3x32x32
```


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os
import matplotlib.pyplot as plt

In [2]:
# ---------------------------------------------------------------------------
# モデル定義
# ---------------------------------------------------------------------------
class ConvAutoencoder(nn.Module):
    """畳み込み層で構成した自己符号化器である．

    エンコーダは stride=2 の畳み込みで一辺を半分ずつに縮め，その分だけ
    チャンネル数を増やしていく（情報を空間方向からチャンネル方向へ移す）．
    デコーダは転置畳み込みでこれをちょうど逆にたどり，元の解像度へ戻す．
    中央の latent_dim が砂時計の最も細い「くびれ」にあたる．
    """

    def __init__(self, in_channels: int = 3, latent_dim: int = 128):
        super().__init__()
        self.in_channels = in_channels

        # 畳み込みの最終出力は 128 チャンネル x 4 x 4 = 2048 次元になる．
        # この値はデコーダ側で潜在表現を特徴マップへ戻すときにも使う．
        self._enc_channels = 128
        self._enc_spatial = 4
        flat_dim = self._enc_channels * self._enc_spatial * self._enc_spatial

        # --- エンコーダ: 3x32x32 -> 128x4x4 ---
        # 各畳み込みの後に BatchNorm（学習の安定化）と ReLU を置く．
        # kernel=3, stride=2, padding=1 で一辺がちょうど半分になる．
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1),  # 32 -> 16
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # 16 -> 8
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),  # 8 -> 4
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        # 特徴マップを平坦化し，全結合層で潜在表現（ボトルネック）へ落とす．
        self.encoder_fc = nn.Linear(flat_dim, latent_dim)

        # --- デコーダ: 潜在表現 -> 3x32x32 ---
        # まず全結合で 2048 次元へ戻し，128x4x4 の特徴マップへ reshape する．
        self.decoder_fc = nn.Linear(latent_dim, flat_dim)

        # 転置畳み込みで一辺を 2 倍ずつに拡大する．
        # output_padding=1 を入れることで，stride=2 の畳み込みの逆を
        # 過不足なく復元できる（4 -> 8 -> 16 -> 32）．
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(
                128, 64, kernel_size=3, stride=2, padding=1, output_padding=1
            ),  # 4 -> 8
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(
                64, 32, kernel_size=3, stride=2, padding=1, output_padding=1
            ),  # 8 -> 16
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(
                32, in_channels, kernel_size=3, stride=2, padding=1, output_padding=1
            ),  # 16 -> 32
            nn.Sigmoid(),  # 画素値を [0, 1] に収める
        )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """入力画像から潜在表現を取り出す．特徴抽出・異常検知に用いる．"""
        h = self.encoder_conv(x)
        h = torch.flatten(h, start_dim=1)  # (N, 128, 4, 4) -> (N, 2048)
        z = self.encoder_fc(h)
        return z

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """潜在表現から画像を復元する．"""
        h = self.decoder_fc(z)
        # (N, 2048) -> (N, 128, 4, 4) へ戻す
        h = h.view(-1, self._enc_channels, self._enc_spatial, self._enc_spatial)
        x_hat = self.decoder_conv(h)
        return x_hat

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decode(self.encode(x))


In [3]:
# ---------------------------------------------------------------------------
# データ
# ---------------------------------------------------------------------------
def build_dataloaders(data_root: str, batch_size: int):
    """CIFAR-10 の学習用・テスト用 DataLoader を構築する．

    ToTensor() により画素値は [0, 1] へ正規化される．畳み込みは画像を
    平坦化せず (N, C, H, W) の形のまま入力できる点が全結合版との違いである．
    """
    transform = transforms.ToTensor()

    train_set = datasets.CIFAR10(
        root=data_root, train=True, download=True, transform=transform
    )
    test_set = datasets.CIFAR10(
        root=data_root, train=False, download=True, transform=transform
    )

    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True, num_workers=2
    )
    test_loader = DataLoader(
        test_set, batch_size=batch_size, shuffle=False, num_workers=2
    )
    return train_loader, test_loader


In [4]:
# ---------------------------------------------------------------------------
# 学習・評価
# ---------------------------------------------------------------------------
def train_one_epoch(model, loader, criterion, optimizer, device) -> float:
    """1 エポック分の学習を行い，平均損失を返す．"""
    model.train()
    running_loss = 0.0
    n_samples = 0

    for images, _ in loader:  # ラベルは使わない（教師なし学習）
        images = images.to(device)  # (N, 3, 32, 32) のまま入力する

        x_hat = model(images)
        loss = criterion(x_hat, images)  # 入力画像自身が復元の目標である

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        n_samples += images.size(0)

    return running_loss / n_samples


@torch.no_grad()
def evaluate(model, loader, criterion, device) -> float:
    """テストデータでの平均復元誤差を返す．"""
    model.eval()
    running_loss = 0.0
    n_samples = 0

    for images, _ in loader:
        images = images.to(device)
        x_hat = model(images)
        loss = criterion(x_hat, images)
        running_loss += loss.item() * images.size(0)
        n_samples += images.size(0)

    return running_loss / n_samples


In [8]:
# ---------------------------------------------------------------------------
# 可視化
# ---------------------------------------------------------------------------
@torch.no_grad()
def save_reconstructions(model, loader, device, out_path: str, n: int = 10):
    """テスト画像（上段）と，その復元結果（下段）を並べて保存する．"""
    model.eval()
    images, _ = next(iter(loader))
    images = images[:n].to(device)
    x_hat = model(images)

    # 表示用に (N, H, W, C) へ並べ替える（matplotlib は HWC を想定する）
    originals = images.cpu().permute(0, 2, 3, 1).numpy()
    reconstructions = x_hat.cpu().permute(0, 2, 3, 1).numpy()

    fig, axes = plt.subplots(2, n, figsize=(n * 1.2, 2.6))
    for i in range(n):
        axes[0, i].imshow(originals[i])
        axes[0, i].axis("off")
        axes[1, i].imshow(reconstructions[i])
        axes[1, i].axis("off")
    fig.suptitle("Top: input   /   Bottom: reconstruction")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"再構成結果を {out_path} に保存した．")


In [10]:
# ---------------------------------------------------------------------------
# メイン
# ---------------------------------------------------------------------------
def main():
    EPOCHS = 30 # 学習エポック数
    BATCH_SIZE = 256 # バッチサイズ
    LR = 1e-3 # 学習率
    LATENT_DIM = 128 # 潜在表現の次元数
    IN_CHANNELS = 3 # 入力チャンネル数

    DATA_ROOT = "./data" # データ保存先
    OUT_DIR = "./outputs" # 成果物の保存先

    os.makedirs(OUT_DIR, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    train_loader, test_loader = build_dataloaders(DATA_ROOT, BATCH_SIZE)

    model = ConvAutoencoder(
        in_channels=IN_CHANNELS, latent_dim=LATENT_DIM
    ).to(device)
    print(model)

    # 損失関数の選択
    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        test_loss = evaluate(model, test_loader, criterion, device)
        print(
            f"[epoch {epoch:3d}/{EPOCHS}] "
            f"train_loss={train_loss:.6f}  test_loss={test_loss:.6f}"
        )

    ckpt_path = os.path.join(OUT_DIR, "cifar10_conv_ae.pth")
    torch.save(model.state_dict(), ckpt_path)
    print(f"モデルを {ckpt_path} に保存した．")

    save_reconstructions(
        model, test_loader, device, os.path.join(OUT_DIR, "reconstruction.png")
    )

    '''
    # 潜在表現の形を確認する
    sample_images, _ = next(iter(test_loader))
    sample_images = sample_images.to(device)
    with torch.no_grad():
        z = model.encode(sample_images)
    print(
        f"潜在表現の形状: {tuple(z.shape)}  "
        f"（3x32x32 = 3072 次元から {LATENT_DIM} 次元へ圧縮された）"
    )
    '''


if __name__ == "__main__":
    main()

使用デバイス: cuda
ConvAutoencoder(
  (encoder_conv): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU(inplace=True)
  )
  (encoder_fc): Linear(in_features=2048, out_features=128, bias=True)
  (decoder_fc): Linear(in_features=128, out_features=2048, bias=True)
  (decoder_conv): Sequential(
    (0): ConvTranspose2d(128, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), output_padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stat